In [1]:
# Installe Unsloth et les dépendances nécessaires
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install sentence-transformers faiss-cpu tqdm rank-bm25 gradio

In [2]:
!pip -q install -U transformers accelerate bitsandbytes \
  "huggingface_hub>=0.34.0,<1.0" \
  sentence-transformers faiss-cpu tqdm rank-bm25 gradio

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"

from huggingface_hub import notebook_login
notebook_login()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.1.2 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,<=4.57.3,>=4.51.3, but you have transformers 4.57.5 which is incompatible.
Mounted at /content/drive


In [3]:
import os
from pathlib import Path

DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"
print("DATA_DIR =", DATA_DIR)
print("Existe ?", os.path.exists(DATA_DIR))

# Lister rapidement le contenu
print("\nContenu (20 premiers):")
print(os.listdir(DATA_DIR)[:20])

# Compter les fichiers JSON (récursif)
json_files = sorted([str(p) for p in Path(DATA_DIR).rglob("*.json")])
json_files_upper = sorted([str(p) for p in Path(DATA_DIR).rglob("*.JSON")])
print("\nNb .json :", len(json_files))
print("Nb .JSON :", len(json_files_upper))

# Montrer un exemple
example_list = json_files or json_files_upper
print("\nExemple:", example_list[0] if example_list else "Aucun JSON trouvé")

DATA_DIR = /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full
Existe ? True

Contenu (20 premiers):
['unsloth_compiled_cache', 'huggingface_tokenizers_cache', 'outputs', 'wandb', 'cleaned_json_full']

Nb .json : 139
Nb .JSON : 0

Exemple: /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/cleaned_json_full/guide_candidat_2025.json


In [4]:
import json

path0 = (json_files or json_files_upper)[0]
with open(path0, "r", encoding="utf-8") as f:
    sample = json.load(f)

print("Fichier:", path0)
print("Type:", type(sample))
if isinstance(sample, dict):
    print("Keys:", list(sample.keys())[:50])
else:
    print("Longueur liste:", len(sample))
    print("Keys du premier item:", list(sample[0].keys())[:50])

Fichier: /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/cleaned_json_full/guide_candidat_2025.json
Type: <class 'dict'>
Keys: ['source', 'url', 'source_file', 'pages', 'sections']


In [5]:
import os, json, re
from pathlib import Path

def norm(x) -> str:
    """
    Normalise n'importe quel type vers une string 'propre'.
    - str -> normalisation whitespace
    - list/dict -> json stringifié
    - autres -> str(...)
    """
    if x is None:
        return ""
    if isinstance(x, str):
        s = x
    elif isinstance(x, (dict, list)):
        s = json.dumps(x, ensure_ascii=False)
    else:
        s = str(x)
    return re.sub(r"\s+", " ", s).strip()

def guess_kind(filename: str) -> str:
    fn = filename.lower()
    if fn.startswith("page_"):
        return "concours"
    if "deroul" in fn or "déroul" in fn or "process" in fn:
        return "general_process"
    if "avantage" in fn or "remuner" in fn or "rémun" in fn or "accompagner" in fn or "carriere" in fn or "carrière" in fn:
        return "general_career"
    if "institut" in fn or "cnrs" in fn:
        return "general_cnrs"
    return "general_other"

In [6]:
def extract_passages(obj: dict, filename: str):
    kind = guess_kind(filename)
    source = obj.get("url") or obj.get("source_url") or obj.get("source_file") or f"local://{filename}"

    passages = []

    # 1) Format "pages": liste de pages/sections
    if isinstance(obj.get("pages"), list):
        for i, p in enumerate(obj["pages"], start=1):
            # p peut être dict ou str
            if isinstance(p, dict):
                sec = p.get("title") or p.get("heading") or p.get("section") or f"Page {i}"
                txt = p.get("text") or p.get("content") or p.get("body") or ""
            else:
                sec = f"Page {i}"
                txt = str(p)
            txt = norm(txt)
            if txt:
                passages.append({
                    "text": txt,
                    "source": source,
                    "section": norm(sec),
                    "doc_id": filename,
                    "title": obj.get("title") or filename,
                    "kind": kind
                })
        return passages

    # 2) Format "institutes": liste (institut CNRS)
    if isinstance(obj.get("institutes"), list):
        for inst in obj["institutes"]:
            if not isinstance(inst, dict):
                continue
            name = inst.get("name") or inst.get("title") or inst.get("acronym") or "Institut"
            desc = inst.get("description") or inst.get("text") or inst.get("content") or ""
            desc = norm(desc)
            if desc:
                passages.append({
                    "text": desc,
                    "source": source,
                    "section": f"Institut: {norm(name)}",
                    "doc_id": filename,
                    "title": obj.get("title") or "Instituts CNRS",
                    "kind": kind
                })
        return passages

    # 3) Format concours / général : champs texte classiques
    title = obj.get("title") or obj.get("intitule") or obj.get("nom") or filename
    # Certains JSON ont des sections structurées
    if isinstance(obj.get("sections"), list):
        for sec in obj["sections"]:
            if not isinstance(sec, dict):
                continue
            sec_title = sec.get("title") or sec.get("heading") or sec.get("section") or "Section"
            sec_text = sec.get("text") or sec.get("content") or sec.get("body") or ""
            sec_text = norm(sec_text)
            if sec_text:
                passages.append({
                    "text": sec_text,
                    "source": source,
                    "section": norm(sec_title),
                    "doc_id": filename,
                    "title": title,
                    "kind": kind
                })
        return passages

    # 4) Fallback texte brut
    text = obj.get("text") or obj.get("content") or obj.get("body") or obj.get("texte") or ""
    text = norm(text)
    if text:
        passages.append({
            "text": text,
            "source": source,
            "section": "Document",
            "doc_id": filename,
            "title": title,
            "kind": kind
        })
        return passages

    # 5) Dernier recours: stringify propre (évite de perdre des infos)
    blob = norm(json.dumps(obj, ensure_ascii=False))
    if blob:
        passages.append({
            "text": blob,
            "source": source,
            "section": "Document (json)",
            "doc_id": filename,
            "title": title,
            "kind": kind
        })
    return passages

In [7]:
json_paths = sorted([str(p) for p in Path(DATA_DIR).rglob("*.json")])

passages = []
kinds_count = {}

for path in json_paths:
    fn = os.path.basename(path)
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict):
        ps = extract_passages(obj, fn)
    elif isinstance(obj, list):
        ps = []
        for item in obj:
            if isinstance(item, dict):
                ps.extend(extract_passages(item, fn))
    else:
        ps = []
    passages.extend(ps)

for p in passages:
    kinds_count[p["kind"]] = kinds_count.get(p["kind"], 0) + 1

print("✅ Passages total:", len(passages))
print("Répartition kinds:", kinds_count)

# exemples
for ex in passages[:3]:
    print("\n---", ex["kind"], "|", ex["doc_id"], "|", ex["section"])
    print("source:", ex["source"])
    print(ex["text"][:250], "...")

✅ Passages total: 170
Répartition kinds: {'general_other': 33, 'general_cnrs': 4, 'concours': 122, 'general_career': 11}

--- general_other | guide_candidat_2025.json | Page 1
source: Guide candidat 2025.pdf
CONCOURS EXTERNES DES PERSONNELS INGÉNIEURS ET TECHNICIENS Le guide du candidat et de la candidate Edition 2025 ...

--- general_other | guide_candidat_2025.json | Page 2
source: Guide candidat 2025.pdf
Direction de la publication : Antoine Petit Direction de la rédaction : Hélène Maury Direction adjointe de la rédaction : Christiane Ename – Laetitia Navarro -Service recrutement et intégration (SeRI) Autrices : Dominique Marx - Emilie Faure - Nathal ...

--- general_other | guide_candidat_2025.json | Page 3
source: Guide candidat 2025.pdf
5 - 6 Pourquoi candidater ? 7 - 8 Le choix des concours 9 - 10 L’inscription 11 Comment concourir ? 12 Les conditions pour concourir 13 - 14 Le déroulement des concours 15-16 Les épreuves 17 La publication des résultats 18 La rémunération 19 RGPD 

In [8]:
import re

def chunk_text(text: str, chunk_size=900, overlap=120):
    sents = re.split(r"(?<=[\.\!\?])\s+", text)
    chunks, cur = [], ""
    for s in sents:
        s = s.strip()
        if not s:
            continue
        if len(cur) + len(s) + 1 <= chunk_size:
            cur = (cur + " " + s).strip()
        else:
            if cur:
                chunks.append(cur)
            # overlap = fin du chunk précédent
            if overlap > 0 and chunks:
                tail = chunks[-1][-overlap:]
                cur = (tail + " " + s).strip()
            else:
                cur = s
    if cur:
        chunks.append(cur)
    return chunks

chunked = []
for p in passages:
    for c in chunk_text(p["text"], chunk_size=900, overlap=120):
        chunked.append({**p, "text": c})

print("✅ Chunks:", len(chunked))

✅ Chunks: 756


In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np, faiss
from tqdm import tqdm

EMBED_ID = "BAAI/bge-m3"
embedder = SentenceTransformer(EMBED_ID)

texts = [c["text"] for c in chunked]

def embed_all(texts, batch_size=64):
    vecs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        v = embedder.encode(texts[i:i+batch_size], normalize_embeddings=True, show_progress_bar=False)
        vecs.append(v)
    return np.vstack(vecs).astype("float32")

emb = embed_all(texts, batch_size=64)

index = faiss.IndexFlatIP(emb.shape[1])  # cosine via vecteurs normalisés
index.add(emb)

print("✅ FAISS size:", index.ntotal, "dim:", emb.shape[1])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


 92%|█████████▏| 11/12 [01:49<00:09,  9.95s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 8.00 GiB. GPU 0 has a total capacity of 14.74 GiB of which 3.48 GiB is free. Process 6348 has 11.26 GiB memory in use. Of the allocated memory 11.13 GiB is allocated by PyTorch, and 8.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from sentence_transformers import CrossEncoder

RERANK_ID = "BAAI/bge-reranker-v2-m3"
reranker = CrossEncoder(RERANK_ID)
print("✅ Reranker chargé:", RERANK_ID)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Reranker chargé: BAAI/bge-reranker-v2-m3


In [ ]:
def retrieve(query: str, k=10, pre_k=60, kind_filter=None):
    qv = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, pre_k)

    cand = []
    for s, idx in zip(scores[0], ids[0]):
        c = chunked[int(idx)]
        if kind_filter and c["kind"] not in kind_filter:
            continue
        cand.append(c)

    if not cand:
        return []

    # rerank
    pairs = [(query, c["text"]) for c in cand]
    rr = reranker.predict(pairs)

    ranked = sorted(zip(rr, cand), key=lambda x: x[0], reverse=True)[:k]
    out = []
    for rr_score, c in ranked:
        out.append({
            "score": float(rr_score),
            "text": c["text"],
            "source": c["source"],
            "section": c["section"],
            "doc_id": c["doc_id"],
            "kind": c["kind"],
        })
    return out

# petit test
res = retrieve("conditions d'accès au concours ingénieur CNRS", k=5, kind_filter={"general_career", "general_cnrs"})
for r in res:
    print(r["score"], r["doc_id"], r["section"])

0.9277984499931335 vous_accompagner.json Page 2
0.832401692867279 vous_accompagner.json Page 2
0.026805903762578964 vous_accompagner.json Page 3
0.02623230591416359 vos_avantages.json Page 4


In [ ]:
# ==========================================
# 1. CHARGEMENT DU MODÈLE POUR L'ENTRAÎNEMENT
# ==========================================
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments

max_seq_length = 2048
dtype = None
load_in_4bit = True

print("⏳ Chargement de Llama 3 via Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Ajout des adaptateurs (LoRA)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# ==========================================
# 2. PRÉPARATION DES DONNÉES (train.jsonl)
# ==========================================
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"},
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# Chargement de ton fichier train.jsonl
dataset = load_dataset("json", data_files = "train.jsonl", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

# ==========================================
# 3. LANCEMENT DE L'ENTRAÎNEMENT (Fine-Tuning)
# ==========================================
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 80, # Entraînement rapide (tu peux mettre 100 si tu veux plus fort)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Démarrage de l'entraînement...")
trainer_stats = trainer.train()
print("🎉 Modèle entraîné avec succès !")

# Passage en mode Utilisation (Inférence)
FastLanguageModel.for_inference(model)

# ==========================================
# 4. NOUVELLE FONCTION DE CHAT
# ==========================================
@torch.inference_mode()
def llama_chat(system: str, user: str, max_new_tokens=450, temperature=0.1):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = temperature,
        do_sample = True
    )

    response = tokenizer.batch_decode(outputs)
    # Nettoyage
    text_response = response[0].split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip()

    return text_response

print("✅ Nouveau système de Chat prêt (Modèle Fine-tuné)")

/tmp/ipython-input-239838928.py:4: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Chargement de Llama 3 via Unsloth...
==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.5.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.104700
2,3.114900
3,3.234300
4,2.872100
5,2.820500
6,2.400000
7,2.243700
8,1.948900
9,1.825800
10,1.634100


wandb: WARNING URL not available in offline run


train/epoch,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇███
train/grad_norm,█▆▆▄▄▄▄▅▄▄█▆▅▆▅▇▄▆▄▅▄▂▃▅▃▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁
train/learning_rate,▁▂▄▅▇████▇▇▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁
train/loss,██▇▆▅▄▄▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,4113707027398656.0
train/epoch,20
train/global_step,80
train/grad_norm,0.20639
train/learning_rate,0.0
train/loss,0.0441


🎉 Modèle entraîné avec succès !
✅ Nouveau système de Chat prêt (Modèle Fine-tuné)


In [ ]:
import re

SYSTEM = """Tu es un agent conversationnel RAG sur des documents CNRS (concours ingénieur).
Règles STRICTES anti-fausses-informations:
- Réponds en français.
- Utilise UNIQUEMENT le CONTEXTE. Ne complète jamais avec ta mémoire.
- Si tu n'es pas sûr à partir du CONTEXTE, dis: "Je ne peux pas répondre de manière fiable avec les documents disponibles."
- Cite tes sources sous forme [n] dans le texte (au moins à la fin de chaque point important).
- Ne fabrique jamais : dates, lieux, conditions, montants, intitulés.
- Ne mets pas de section "Sources" (elle sera ajoutée après).
"""

def format_context(passages, max_chars=420):
    lines=[]
    for i,p in enumerate(passages,1):
        excerpt = p["text"][:max_chars].strip() + ("..." if len(p["text"])>max_chars else "")
        lines.append(f"[{i}] ({p['source']} — {p['section']})\n{excerpt}")
    return "\n\n".join(lines)

def build_sources(passages, max_items=10):
    lines=["Sources:"]
    seen=set()
    for i,p in enumerate(passages,1):
        key=(p["source"], p["section"])
        if key in seen:
            continue
        seen.add(key)
        lines.append(f"- [{i}] {p['source']} | Section: {p['section']}")
        if len(seen) >= max_items:
            break
    return "\n".join(lines)

def detect_mode(q: str):
    ql = q.lower()
    if any(w in ql for w in ["je suis", "profil", "compétence", "competence", "qualif", "qualification", "stack", "candidat", "orienter", "quel concours"]):
        return "orient"
    if any(w in ql for w in ["carrière", "carriere", "grade", "rémun", "remun", "avantage", "prime", "branches", "métier", "metier"]):
        return "career"
    return "concours_info"

def answer_question(question: str, k=10, min_score=0.15):
    mode = detect_mode(question)

    if mode == "career":
        kind_filter = {"general_career","general_cnrs"}
    elif mode == "orient":
        kind_filter = {"concours"}
    else:
        kind_filter = None

    passages = retrieve(question, k=k, kind_filter=kind_filter)
    if not passages or passages[0]["score"] < min_score:
        return "Je ne peux pas répondre de manière fiable avec les documents disponibles.", []

    ctx = format_context(passages)

    user = f"""MODE: {mode}
QUESTION:
{question}

CONTEXTE:
{ctx}

Consigne de sortie:
- Si MODE=orient : propose 3 à 5 concours pertinents (titre/identifiant) + raison + citations.
- Sinon : réponds de façon structurée (puces) + citations.
"""
    ans = llama_chat(SYSTEM, user)
    final = ans.strip() + "\n" + build_sources(passages)
    return final, passages

In [ ]:
# Interface chat basique (Notebook)

chat_history = []  # optionnel si tu veux stocker les échanges ici

def chat_loop():
    print("=== Chatbot RAG CNRS (Notebook) ===")
    print("Commandes: /quit pour quitter, /reset pour vider l'historique\n")

    while True:
        q = input("Vous: ").strip()
        if not q:
            continue

        if q.lower() in ["/quit", "quit", "exit"]:
            print("Fin du chat.")
            break

        if q.lower() in ["/reset", "reset"]:
            chat_history.clear()
            print("✅ Historique réinitialisé.\n")
            continue

        ans, _ = answer_question(q)
        chat_history.append(("Vous", q))
        chat_history.append(("Assistant", ans))

        print("\nAssistant:\n" + ans + "\n")

chat_loop()

=== Chatbot RAG CNRS (Notebook) ===
Commandes: /quit pour quitter, /reset pour vider l'historique

Vous: bonjour


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Assistant:
MODE: concours_info
QUESTION:
bonjour

Je ne peux pas répondre de manière fiable avec les documents disponibles. En tant que modèle de support, je n'ai accès qu'aux questions et contextes fournis. Pour obtenir des recommandations de concours, veuillez me fournir les détails de votre profil (compétences, expérience, objectifs) ou demander une orientation générale.
Sources:
- [1] page_015.html | Section: Document (json)
- [2] page_066.html | Section: Document (json)
- [3] page_053.html | Section: Document (json)
- [4] page_018.html | Section: Document (json)
- [5] page_025.html | Section: Document (json)
- [6] page_061.html | Section: Document (json)
- [7] page_005.html | Section: Document (json)
- [8] page_077.html | Section: Document (json)
- [10] page_059.html | Section: Document (json)

Vous: donne moi une recette de pizza 

Assistant:
Je ne peux pas répondre de manière fiable avec les documents disponibles.

Vous: Quelles sont les conditions pour passer le concours ingén

OutOfMemoryError: CUDA out of memory. Tried to allocate 762.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 596.12 MiB is free. Process 7132 has 14.16 GiB memory in use. Of the allocated memory 11.91 GiB is allocated by PyTorch, and 2.09 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import re

def detect_smalltalk(text: str):
    t = text.lower().strip()

    greet = {"bonjour","salut","hello","bonsoir","coucou"}
    bye = {"au revoir","aurevoir","bye","à bientôt","a bientot","bonne journée","bonne soiree","bonne soirée"}
    thanks = {"merci","merci beaucoup","thx","thanks","je te remercie"}

    if t in greet or re.match(r"^(bonjour|salut|hello|bonsoir)\b", t):
        return "greet"
    if t in thanks or re.match(r"^merci\b", t):
        return "thanks"
    if t in bye or re.match(r"^(au revoir|bye)\b", t):
        return "bye"
    if t in {"aide","help","?"}:
        return "help"
    return None

def smalltalk_response(intent: str):
    if intent == "greet":
        return ("Bonjour 👋 Je peux t’aider à :\n"
                "- trouver des concours ingénieur CNRS selon tes compétences\n"
                "- expliquer un concours précis\n"
                "- donner des infos sur les carrières (grades, avantages, rémunération si présent)\n"
                "- expliquer le déroulement des concours (si présent dans les docs)\n\n"
                "Dis-moi ton profil (compétences, domaine) ou le concours qui t’intéresse.")
    if intent == "thanks":
        return "Avec plaisir ! Si tu veux, décris ton profil (compétences/expérience) et je te propose des concours pertinents."
    if intent == "bye":
        return "Au revoir ! N’hésite pas à revenir si tu as d’autres questions sur les concours CNRS."
    if intent == "help":
        return ("Tu peux me demander par exemple :\n"
                "- « Je suis ingénieur data, quels concours me correspondent ? »\n"
                "- « Donne-moi les missions/compétences du concours page_072 »\n"
                "- « Quels sont les avantages à travailler au CNRS ? »\n"
                "- « Quelles sont les phases du concours ? »")
    return "Je suis là 🙂"

In [ ]:
def rewrite_query(user_question: str, history_pairs, max_len=220):
    ctx = "\n".join([f"User: {u}\nAssistant: {a}" for u,a in history_pairs[-2:]])
    prompt = f"""Tu es un assistant qui reformule des questions pour un moteur de recherche documentaire.
Reformule la QUESTION en une requête autonome, courte et précise, en français.
N'ajoute pas d'information. Ne réponds pas à la question.

HISTORIQUE (optionnel):
{ctx}

QUESTION:
{user_question}

Requête reformulée:"""

    msgs = [
        {"role":"system","content":"Tu reformules des requêtes."},
        {"role":"user","content":prompt},
    ]
    enc = llm_tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    input_ids = enc["input_ids"].to(llm.device)
    attention_mask = enc["attention_mask"].to(llm.device)

    out = llm.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=80,
        do_sample=False,   # ✅ greedy decoding
        eos_token_id=llm_tok.eos_token_id,
        pad_token_id=llm_tok.pad_token_id,
    )

    gen = out[0][input_ids.shape[-1]:]
    q = llm_tok.decode(gen, skip_special_tokens=True).strip()
    return q.replace("\n"," ")[:max_len]

In [ ]:
from collections import defaultdict

def group_by_doc(passages):
    grouped = defaultdict(list)
    for p in passages:
        grouped[p["doc_id"]].append(p)
    return grouped

In [ ]:
chat_pairs = []  # [(user, assistant), ...]

SYSTEM_STRICT = """Tu es un agent conversationnel RAG sur des documents CNRS (concours ingénieur).
Règles STRICTES:
- Réponds en français, avec des phrases correctes et un ton naturel.
- Utilise UNIQUEMENT le CONTEXTE fourni. Ne complète jamais avec ta mémoire.
- Si tu ne peux pas répondre de façon fiable à partir du contexte, dis-le clairement.
- Ne JAMAIS inventer : dates, lieux, conditions, montants, intitulés.
- Mets des citations [n] dans le texte pour les infos importantes.
- Ne mets pas "Sources:" (ajouté automatiquement).
"""

def format_context(passages, max_chars=420):
    lines=[]
    for i,p in enumerate(passages,1):
        excerpt = p["text"][:max_chars].strip() + ("..." if len(p["text"])>max_chars else "")
        lines.append(f"[{i}] ({p['source']} — {p['section']} — {p['doc_id']})\n{excerpt}")
    return "\n\n".join(lines)

def build_sources_used(answer_text, passages, max_items=12):
    used = sorted(set(int(n) for n in re.findall(r"\[(\d+)\]", answer_text)))
    lines=["Sources:"]
    seen=set()
    count=0
    for n in used:
        if 1 <= n <= len(passages):
            p = passages[n-1]
            key=(p["source"], p["section"])
            if key in seen:
                continue
            seen.add(key)
            lines.append(f"- [{n}] {p['source']} | Section: {p['section']}")
            count += 1
            if count >= max_items:
                break
    # si aucune citation, on met quand même les 3 meilleures sources
    if len(lines) == 1 and passages:
        for i,p in enumerate(passages[:3],1):
            lines.append(f"- [{i}] {p['source']} | Section: {p['section']}")
    return "\n".join(lines)

def detect_mode(q: str):
    ql = q.lower()
    if any(w in ql for w in ["je suis", "profil", "compétence", "competence", "qualification", "orient", "quel concours", "correspond"]):
        return "orient"
    if any(w in ql for w in ["carrière","carriere","grade","rémun","remun","avantage","prime","branches","métier","metier"]):
        return "career"
    if any(w in ql for w in ["phase","dérou","derou","audition","jury","calendrier","date","lieu","épreuve","conditions"]):
        return "process_or_rules"
    return "concours_info"

def chatbot_respond(user_text: str, k=12, min_score=0.15):
    global chat_pairs

    # 1) smalltalk
    intent = detect_smalltalk(user_text)
    if intent:
        ans = smalltalk_response(intent)
        chat_pairs.append((user_text, ans))
        return ans

    # 2) rewrite query for better retrieval
    rq = rewrite_query(user_text, chat_pairs)

    # 3) mode -> kind_filter (avec ce qu'on a dans tes JSON)
    mode = detect_mode(user_text)
    if mode == "career":
        kind_filter = {"general_career", "general_cnrs"}
    elif mode == "orient":
        kind_filter = {"concours"}
    else:
        kind_filter = None

    # 4) retrieve
    passages = retrieve(rq, k=k, kind_filter=kind_filter)

    if not passages or passages[0]["score"] < min_score:
        ans = ("Je ne peux pas répondre de manière fiable avec les documents disponibles.\n"
               "Tu peux reformuler, ou me donner le nom/ID du concours (ex: page_072) si tu en as un.")
        chat_pairs.append((user_text, ans))
        return ans

    ctx = format_context(passages)

    # 5) génération (réponse structurée)
    if mode == "orient":
        user_prompt = f"""Tu dois aider à orienter vers les concours pertinents.
À partir du CONTEXTE, propose 3 à 5 concours (doc_id) maximum.
Pour chacun: titre si visible, pourquoi (compétences/mission) + citations [n].
Si l'info est absente, dis-le.

QUESTION:
{user_text}

CONTEXTE:
{ctx}
"""
    else:
        user_prompt = f"""Réponds de façon claire et structurée (phrases correctes, puces si utile).
Si la réponse est partielle, ajoute une section "Limites" (1-2 lignes).
QUESTION:
{user_text}

CONTEXTE:
{ctx}
"""

    ans = llama_chat(SYSTEM_STRICT, user_prompt, max_new_tokens=520, temperature=0.1)
    final = ans.strip() + "\n" + build_sources_used(ans, passages)

    chat_pairs.append((user_text, final))
    return final

In [ ]:
def chat_loop():
    print("=== Chatbot RAG CNRS (Notebook) ===")
    print("Commandes: /quit, /reset\n")
    while True:
        q = input("Vous: ").strip()
        if not q:
            continue
        if q.lower() in ["/quit", "quit", "exit"]:
            print("Fin du chat.")
            break
        if q.lower() in ["/reset", "reset"]:
            chat_pairs.clear()
            print("✅ Historique réinitialisé.\n")
            continue

        ans = chatbot_respond(q)
        print("\nAssistant:\n" + ans + "\n")

chat_loop()

=== Chatbot RAG CNRS (Notebook) ===
Commandes: /quit, /reset

Vous: bonjour

Assistant:
Bonjour 👋 Je peux t’aider à :
- trouver des concours ingénieur CNRS selon tes compétences
- expliquer un concours précis
- donner des infos sur les carrières (grades, avantages, rémunération si présent)
- expliquer le déroulement des concours (si présent dans les docs)

Dis-moi ton profil (compétences, domaine) ou le concours qui t’intéresse.

Vous: si je suis ingénieur biologiste le  concours 47 me convient ? 


NameError: name 'llm_tok' is not defined